# 📦 InFi-Check — Prepare SFT Dataset (Xuất JSONL)

Notebook này thực hiện **Bước 6** — bước cuối cùng trong pipeline InFi-Check:
```
new_supported_summary/ + new_reference/ + short_error_dataset/
        ↓  [prepare_dataset_base]
sft_dataset/jsonl/
    ├── summary_sft_train_pos1neg2_with_ref.jsonl
    ├── summary_sft_valid_with_ref.jsonl
    └── summary_sft_test_with_ref.jsonl
```

| Bước | File | Mô tả |
|------|------|-------|
| 1 | `dataset_*.py` | Làm sạch data |
| 2 | ✅ | Upload document lên Drive |
| 3 | ✅ `summary_gen_pipeline.ipynb` | Sinh tóm tắt |
| 4 | ✅ `eval_and_reference_gen_pipeline.ipynb` | Refine + tìm câu hỗ trợ |
| 5 | ✅ `structured_dataset_gen_pipeline.ipynb` | Sinh error data |
| **6** | **`prepare_dataset_pipeline.ipynb`** | **← Notebook này** |

---

**Notebook này làm gì?**
- Đọc **positive data**: summary đúng + câu hỗ trợ từ document (`new_supported_summary/` + `new_reference/`)
- Đọc **negative data**: summary có lỗi nhân tạo (`short_error_dataset/`)
- Ghép và chia thành **train / valid / test** → xuất file `.jsonl` để fine-tune LLM

**Thay đổi so với phiên bản cũ:**
- ✅ Positive output có **reasoning đầy đủ** (câu tóm tắt ↔ câu gốc trong document)
- ✅ Tỷ lệ pos:neg đổi từ **1:5 → 1:2** (cân bằng hơn, tránh bias YES)
- ✅ Filter evidence yếu bằng **confidence score** từ Bước 4
- ✅ **Domain cap** — mỗi domain tối đa 30% tổng dataset

**Format mỗi sample:**
```
<|start_header_id|>: [Task + Document + Summary]
<|end_header_id|>: [Phân tích lỗi HOẶC xác nhận đúng kèm reasoning]
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Mount Google Drive & kiểm tra cấu trúc thư mục

In [ ]:


import os

# ================================================================
# ✏️  CHỈNH SỬA: đường dẫn thư mục trên Drive
# ================================================================
DATASET_ROOT = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset'
PROMPT_ROOT  = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/training_dataset_construct'
OUTPUT_ROOT  = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/training_dataset_construct'

DOCUMENT_PATH  = os.path.join(DATASET_ROOT, 'document')
SUMMARY_PATH   = os.path.join(DATASET_ROOT, 'new_supported_summary')
REFERENCE_PATH = os.path.join(DATASET_ROOT, 'new_reference')
ERROR_PATH     = os.path.join(DATASET_ROOT, 'short_error_dataset')
SFT_PATH       = os.path.join(OUTPUT_ROOT,  'sft_dataset', 'jsonl')
PROMPT_FILE    = os.path.join(PROMPT_ROOT,  'prompt', 'sft_prompt.txt')

os.makedirs(SFT_PATH, exist_ok=True)

# Kiểm tra
def count_files(folder, ext='.txt'):
    if not os.path.exists(folder): return 0
    return len([f for f in os.listdir(folder) if f.endswith(ext)])

def count_files_recursive(folder):
    total = 0
    for _, _, files in os.walk(folder):
        total += len(files)
    return total

n_doc      = count_files(DOCUMENT_PATH)
n_summary  = count_files(SUMMARY_PATH)
n_ref      = count_files(REFERENCE_PATH, '.json')
n_error    = count_files_recursive(ERROR_PATH) if os.path.exists(ERROR_PATH) else 0
n_err_docs = len(os.listdir(ERROR_PATH)) if os.path.exists(ERROR_PATH) else 0

print(f'📂 document/              : {n_doc} files')
print(f'📂 new_supported_summary/ : {n_summary} files')
print(f'📂 new_reference/         : {n_ref} files')
print(f'📂 short_error_dataset/   : {n_err_docs} docs | {n_error} error files tổng cộng')
print(f'📂 Output sft_dataset/    : {SFT_PATH}')
print(f'📄 Prompt file            : {PROMPT_FILE}')
print()
if not os.path.exists(PROMPT_FILE):
    print('⚠️  KHÔNG TÌM THẤY prompt/sft_prompt.txt — kiểm tra lại PROMPT_ROOT')
else:
    print('✅ Tất cả thư mục sẵn sàng')

📂 document/              : 373 files
📂 new_supported_summary/ : 418 files
📂 new_reference/         : 418 files
📂 short_error_dataset/   : 514 docs | 514 error files tổng cộng
📂 Output sft_dataset/    : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/training_dataset_construct/sft_dataset/jsonl
📄 Prompt file            : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/training_dataset_construct/prompt/sft_prompt.txt

✅ Tất cả thư mục sẵn sàng


## 2. Cài đặt thư viện

In [ ]:
!pip install -q jsonlines

## 3. Cấu hình & đọc prompt SFT

In [ ]:
import re
import json
import random
import jsonlines

random.seed(312)

# ── Mapping loại lỗi ──────────────────────────────────────────────────
ERROR_TYPE_DICT = {
    'predicate':             'Predicate Error',
    'entity':                'Entity Error',
    'circumstance':          'Circumstance Error',
    'co-reference':          'Co-reference Error',
    'discourse link':        'Discourse Link Error',
    'extrinsic':             'Extrinsic Error',
    'extrinsic error':       'Extrinsic Error',
    'extrinsic information': 'Extrinsic Error',
}

# Số mẫu tối đa mỗi loại lỗi muốn giữ (dùng để cân bằng)
ERROR_SAMPLE_NUM = {
    'Predicate Error':     2,
    'Entity Error':        2,
    'Circumstance Error':  1,
    'Co-reference Error':  2,
    'Discourse Link Error':1,
    'Extrinsic Error':     1,
}

# ── Tỷ lệ pos:neg trong train set ────────────────────────────────────
# 1:2 = cân bằng tốt cho 7000+ bài; tăng lên 3 nếu muốn model nhạy hơn với lỗi
NEG_PER_POS = 2

# ── Ngưỡng confidence từ voting Bước 4 ───────────────────────────────
# Sample có confidence < ngưỡng này bị bỏ qua (evidence không đáng tin)
MIN_CONFIDENCE = 0.67  # ít nhất 2/3 model đồng ý

# ── Domain map & cap (cho dataset 7000+ bài đa nguồn) ────────────────
DOMAIN_MAP = {
    # Tech
    'Cong_nghe': 'tech',      'So_hoa': 'tech',
    # Business / Economy
    'Kinh_doanh': 'business', 'Kinh_te': 'business',
    'Bat_dong_san': 'business','Thi_truong': 'business',
    # Legal / Politics
    'Phap_luat': 'legal',     'Chinh_tri': 'politics',
    'Cong_doan': 'politics',
    # Lifestyle
    'Doi_song': 'lifestyle',  'Suc_khoe': 'health',
    'Du_lich': 'lifestyle',   'Am_thuc': 'lifestyle',
    'Tam_su': 'lifestyle',    'Y_kien': 'lifestyle',
    # Entertainment / Culture
    'Giai_tri': 'entertainment','Van_hoa': 'culture',
    'Truyen_hinh': 'entertainment',
    # Sports
    'The_thao': 'sports',     'Oto_xe_may': 'sports',
    # World / Society
    'The_gioi': 'world',      'Xa_Hoi': 'society',
    'Thoi_su': 'society',     'Chong_Dien_Bien_Hoa_Binh': 'society',
    # Science / Education / Health
    'Khoa_hoc': 'science',    'Giao_duc': 'education',
    'Y_te': 'health',         'Moi_truong': 'science',
    # ✏️  Thêm folder mới tại đây
}
MAX_DOMAIN_RATIO = 0.30  # tối đa 30% mỗi domain

# ── Đọc prompt SFT ───────────────────────────────────────────────────
with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
    SFT_INPUT_FORMAT = f.read()

print(f'✅ Đã load prompt SFT ({len(SFT_INPUT_FORMAT)} ký tự)')
print(f'   6 loại lỗi : {list(ERROR_SAMPLE_NUM.keys())}')
print(f'   NEG_PER_POS: {NEG_PER_POS}  (tỷ lệ pos:neg = 1:{NEG_PER_POS})')
print(f'   MIN_CONF   : {MIN_CONFIDENCE}')


✅ Đã load prompt SFT (1410 ký tự)
   6 loại lỗi: ['Predicate Error', 'Entity Error', 'Circumstance Error', 'Co-reference Error', 'Discourse Link Error', 'Extrinsic Error']


## 4. Định nghĩa các hàm xử lý dữ liệu

In [ ]:
# ── Bộ đếm thống kê (global) ─────────────────────────────────────────
empty_counter = {k: 0 for k in ERROR_SAMPLE_NUM}
pos_counter = pos_num = 0
neg_counter = neg_num = 0
low_conf_skipped = 0  # đếm số câu bị skip do confidence thấp

# Các prefix category cần bỏ khi tìm document gốc
CATEGORY_PREFIXES = [
    'Cong_nghe_', 'So_hoa_', 'Doi_song_',
    'Phap_luat_', 'Giai_tri_', 'Kinh_doanh_',
]

def strip_category_prefix(title: str) -> str:
    for prefix in CATEGORY_PREFIXES:
        if title.startswith(prefix):
            return title[len(prefix):]
    return title


def get_domain(doc_name: str) -> str:
    for prefix, domain in DOMAIN_MAP.items():
        if doc_name.startswith(prefix + '_'):
            return domain
    return 'other'


def make_sample(input_text: str, output_text: str) -> dict:
    return {"text": f"<|start_header_id|>:{input_text}\n<|end_header_id|>:{output_text}"}


# ─────────────────────────────────────────────────────────────────────
# Sinh output cho negative sample (không thay đổi)
# ─────────────────────────────────────────────────────────────────────
def generate_negative_output(error_dict: dict) -> str:
    add_mark         = True
    modified_element = error_dict['modified element']
    explanation      = error_dict.get('explanation', '')
    wrong_info       = error_dict.get('wrong information', '')

    if (
        'The meaning has not been altered' in explanation
        or 'No wrong information' in wrong_info
        or 'no wrong information' in wrong_info
    ):
        return ''

    error_type = error_dict['error type']

    if error_type == 'Co-reference Error':
        if 'The subject of the new sentence is' in modified_element:
            modified_element = modified_element.replace('The subject of the new sentence is', '').strip()
        elif 'The new pronoun' in modified_element:
            modified_element = modified_element.replace('The new pronoun', '').strip()

    elif error_type == 'Circumstance Error':
        for pattern in [
            r"The new (\w+?) (?:(['\"].*?['\"])|([^\s]+(?:\s+[^\s]+)*)) used to replace the original \1 .+",
            r"The new circumstance ([^']+|'[^']*') used to replace",
            r"The new (.+?) used to replace",
        ]:
            match = re.match(pattern, modified_element) or re.search(pattern, modified_element)
            if match:
                groups = [g for g in match.groups() if g]
                if groups:
                    modified_element = groups[-1]
                    if 'used to replace' in pattern and 'circumstance' not in pattern:
                        modified_element = modified_element[0].lower() + modified_element[1:]
                        if not modified_element.startswith('the'):
                            modified_element = 'the ' + modified_element
                        add_mark = False
                break

    specific_begin = ' Specifically, '
    if not modified_element:
        if 'remove' in error_dict.get('modification explanation', '').lower():
            specific_begin = ''
        else:
            print(f'  [WARN] Empty modified element: {error_dict.get("modified text", "")[:60]}')
    elif len(modified_element) >= 0.9 * len(error_dict.get('modified text', modified_element)):
        specific_begin   = ''
        modified_element = ''
    else:
        if modified_element.endswith('.'):
            modified_element = modified_element[:-1]
        if add_mark and not (modified_element.startswith("'") and modified_element.endswith("'")):
            modified_element = f"'{modified_element}'"
        modified_element += '.'

    original_text = error_dict.get('original text in summary', '')
    method        = error_dict.get('method', '')

    if method == 'merging sentences':
        if 'Sentence 1:' in original_text and 'Sentence 2:' in original_text:
            original_text = original_text.replace('Sentence 1:', '').strip()
            original_text = ' '.join(s.strip() for s in original_text.split('Sentence 2:'))
    elif method == 'swapping numbers':
        for marker in ('The sentence', 'Sentence in'):
            if marker in original_text:
                parts = original_text.split(marker)
                if len(parts) == 2:
                    original_text = parts[0]
                break
        if ':' in original_text:
            prefix = original_text[:original_text.find(':')]
            if 'The original' in prefix or 'Original' in prefix:
                original_text = original_text[original_text.find(':'):].strip()

    return (
        f"The following part is not supported by the origin document:\n"
        f"- Location: The error occurs in the following sentence: {error_dict['modified text']}"
        f"{specific_begin}{modified_element}\n"
        f"- Explanation: {wrong_info}\n"
        f"- Correction: {original_text}\n"
        f"- Error Type: {error_type}\n"
        f"Therefore, the answer is YES."
    )


# ─────────────────────────────────────────────────────────────────────
# Đọc tất cả error file của 1 document (không thay đổi)
# ─────────────────────────────────────────────────────────────────────
def prepare_negative_data(document: str, title: str) -> list:
    global neg_counter, neg_num

    error_folder = os.path.join(ERROR_PATH, title)
    if not os.path.exists(error_folder):
        return []

    negative_set  = []
    error_counter = {k: [] for k in ERROR_SAMPLE_NUM}

    for root, dirs, files in os.walk(error_folder):
        for fname in files:
            if not fname.endswith('.txt'):
                continue
            fpath         = os.path.join(root, fname)
            relative_path = os.path.relpath(fpath, error_folder).replace('\\\\', '/')
            path_parts    = relative_path.split('/')

            try:
                with open(fpath, 'r', encoding='utf-8') as f:
                    error_dict = eval(f.read())
            except Exception as e:
                print(f'  [WARN] Không đọc được {fpath}: {e}')
                continue

            raw_key = path_parts[-3].lower() if len(path_parts) >= 3 else ''
            matched = None
            for k, v in ERROR_TYPE_DICT.items():
                if k in raw_key or raw_key in k:
                    matched = v
                    break
            if matched is None:
                print(f'  [WARN] Không nhận ra error type: "{raw_key}" ({relative_path})')
                continue

            error_dict['error type'] = matched
            error_dict['method']     = path_parts[-2] if len(path_parts) >= 2 else 'unknown'

            input_text  = f"{SFT_INPUT_FORMAT}\nDocument:\n{document}\nSummary:\n{error_dict['full text of modified summary']}"
            output_text = generate_negative_output(error_dict)

            if output_text:
                neg_counter += len(output_text.split())
                neg_num     += 1
                error_counter[matched].append(len(negative_set))
                negative_set.append([make_sample(input_text, output_text), matched])

    empty_types  = [k for k, v in error_counter.items() if len(v) == 0]
    double_types = [k for k, v in error_counter.items() if len(v) >= 2]
    random.shuffle(double_types)
    for i, empty_type in enumerate(empty_types):
        if i >= len(double_types):
            break
        donor_idx  = random.choice([0, 1])
        sample_idx = error_counter[double_types[i]][donor_idx]
        negative_set[sample_idx][1] = empty_type
        empty_counter[empty_type] += 1

    random.shuffle(negative_set)
    return negative_set


# ─────────────────────────────────────────────────────────────────────
# ✅ SỬA: Sinh positive sample với reasoning đầy đủ
# - Mỗi câu tóm tắt được ánh xạ tường minh đến câu gốc trong document
# - Filter câu có confidence thấp (evidence không đáng tin)
# ─────────────────────────────────────────────────────────────────────
def prepare_positive_data(document: str, summary: str, reference: dict) -> dict | None:
    global pos_counter, pos_num, low_conf_skipped

    input_text  = f"{SFT_INPUT_FORMAT}\nDocument:\n{document}\nSummary:\n{summary}"
    output_lines = []
    skipped_sents = 0

    for i, item in enumerate(reference['find_support_result']):
        sent  = item.get('summary sentence', '')
        evids = item.get('reference', item.get('sentences from the document', []))
        conf  = item.get('confidence', 1.0)  # default 1.0 cho data cũ chưa có field này

        # Filter evidence yếu — bỏ qua câu có confidence thấp
        if conf < MIN_CONFIDENCE:
            low_conf_skipped += 1
            skipped_sents    += 1
            print(f'  [LOW_CONF {conf:.2f}] skip: "{sent[:60]}"')
            continue

        if len(evids) == 1:
            output_lines.append(
                f'Sentence {i+1}: "{sent}"\n'
                f'→ Supported by: "{evids[0]}"'
            )
        else:
            joined = '\n  '.join(f'"{e}"' for e in evids)
            output_lines.append(
                f'Sentence {i+1}: "{sent}"\n'
                f'→ Supported by multiple sentences:\n  {joined}'
            )

    # Nếu tất cả câu bị skip → bỏ toàn bộ document này
    if not output_lines:
        print(f'  [SKIP_DOC] Tất cả {skipped_sents} câu đều có confidence thấp — bỏ document')
        return None

    output_text = '\n\n'.join(output_lines)
    output_text += (
        '\n\nAll sentences in the summary are directly supported '
        'by the original document.\n'
        'Therefore, the answer is NO.'
    )

    pos_counter += len(output_text.split())
    pos_num     += 1
    return make_sample(input_text, output_text)


# ─────────────────────────────────────────────────────────────────────
# Load toàn bộ data với domain cap
# ─────────────────────────────────────────────────────────────────────
def prepare_sft_data() -> dict:
    full_data    = {}
    skip_count   = 0
    domain_count = {}

    summary_files = sorted([f for f in os.listdir(SUMMARY_PATH) if f.endswith('.txt')])
    total_docs    = len(summary_files)
    max_per_domain = int(total_docs * MAX_DOMAIN_RATIO)

    print(f'Đang load {total_docs} supported summary...')
    print(f'Domain cap: tối đa {max_per_domain} doc/domain ({MAX_DOMAIN_RATIO*100:.0f}% × {total_docs})')

    for summary_filename in summary_files:
        title  = summary_filename.replace('_supported_summary.txt', '')
        domain = get_domain(title)

        # Domain cap — bỏ qua nếu domain đã đủ
        if domain_count.get(domain, 0) >= max_per_domain:
            skip_count += 1
            continue

        ref_file = os.path.join(REFERENCE_PATH, f'{title}_ref.json')
        doc_file = os.path.join(DOCUMENT_PATH, f'{title}.txt')
        if not os.path.exists(doc_file):
            stripped = strip_category_prefix(title)
            doc_file = os.path.join(DOCUMENT_PATH, f'{stripped}.txt')

        if not os.path.exists(doc_file):
            skip_count += 1
            print(f'  [SKIP] Không tìm thấy document: {title}')
            continue

        with open(os.path.join(SUMMARY_PATH, summary_filename), 'r', encoding='utf-8') as f:
            summary = f.read()
        with open(doc_file, 'r', encoding='utf-8') as f:
            document = f.read()

        neg_data = prepare_negative_data(document, title)

        if os.path.exists(ref_file):
            with open(ref_file, 'r', encoding='utf-8') as f:
                reference = json.load(f)
            pos_sample = prepare_positive_data(document, summary, reference)
            if pos_sample is not None:
                full_data[title] = {'positive': pos_sample, 'negative': neg_data}
                domain_count[domain] = domain_count.get(domain, 0) + 1
            elif neg_data:
                full_data[title] = {'negative': neg_data}
                domain_count[domain] = domain_count.get(domain, 0) + 1
        else:
            if neg_data:
                full_data[title] = {'negative': neg_data}
                domain_count[domain] = domain_count.get(domain, 0) + 1

    print(f'\n✅ Load xong {len(full_data)} document  |  bị SKIP: {skip_count}')
    print(f'   Low confidence sentences skipped: {low_conf_skipped}')
    print(f'   empty_counter (loại lỗi bị mượn): {empty_counter}')
    print(f'   Domain distribution:')
    for d, cnt in sorted(domain_count.items(), key=lambda x: -x[1]):
        print(f'     {d:<15} {cnt:>5}  ({cnt/len(full_data)*100:.1f}%)')
    return full_data


# ─────────────────────────────────────────────────────────────────────
# Tự động tính kích thước split (không thay đổi)
# ─────────────────────────────────────────────────────────────────────
def calc_split(total_size: int):
    if total_size >= 300:
        valid_size = test_size = 100
    elif total_size >= 30:
        valid_size = test_size = max(1, total_size // 10)
    else:
        valid_size = test_size = max(1, total_size // 7)
    train_size = max(0, total_size - valid_size - test_size)
    print(f'  split => train={train_size}  valid={valid_size}  test={test_size}')
    return train_size, valid_size, test_size


# ─────────────────────────────────────────────────────────────────────
# ✅ SỬA: shuffle_and_select — tỷ lệ pos:neg dùng NEG_PER_POS,
#   cân bằng error type trong neg được chọn
# ─────────────────────────────────────────────────────────────────────
def shuffle_and_select_data(full_data: dict, round_num: int = 1):
    sft_train = {i: [] for i in range(round_num)}
    sft_valid, sft_test = [], []

    total_size = len(full_data)
    print(f'total size: {total_size}')
    train_size, valid_size, test_size = calc_split(total_size)

    keys = list(full_data.keys())
    random.shuffle(keys)
    train_data = {k: full_data[k] for k in keys[:train_size]}
    valid_data = {k: full_data[k] for k in keys[train_size:train_size + valid_size]}
    test_data  = {k: full_data[k] for k in keys[train_size + valid_size:]}

    # ── Train: tỷ lệ 1:NEG_PER_POS, cân bằng error type ──────────────
    for rnd in range(round_num):
        pos_list = []
        neg_pool = []  # (neg_sample, error_type)

        for title in train_data:
            if 'positive' in train_data[title]:
                pos_list.append(train_data[title]['positive'])
            for neg_data, error_type in train_data[title]['negative']:
                neg_pool.append((neg_data, error_type))

        n_pos       = len(pos_list)
        n_neg_target= n_pos * NEG_PER_POS

        # Cân bằng error type trong neg được chọn
        random.shuffle(neg_pool)
        n_types     = len(ERROR_SAMPLE_NUM)
        quota       = max(1, n_neg_target // n_types)
        error_count = {k: 0 for k in ERROR_SAMPLE_NUM}
        selected_neg = []

        for neg_data, error_type in neg_pool:
            if error_count.get(error_type, 0) < quota:
                selected_neg.append(neg_data)
                error_count[error_type] = error_count.get(error_type, 0) + 1
            if len(selected_neg) >= n_neg_target:
                break

        sft_train[rnd] = pos_list + selected_neg
        random.shuffle(sft_train[rnd])
        print(f'  Round {rnd}: {n_pos} pos + {len(selected_neg)} neg '
              f'(ratio 1:{len(selected_neg)//max(n_pos,1)}) | error dist: {error_count}')

    # ── Valid/Test: lấy tất cả pos + neg rồi cân bằng 1:1 ────────────
    for split_data, split_list in [(valid_data, sft_valid), (test_data, sft_test)]:
        pos_samples = []
        neg_samples = []
        for title in split_data:
            if 'positive' in split_data[title]:
                pos_samples.append(split_data[title]['positive'])
            for neg_data, _ in split_data[title]['negative']:
                neg_samples.append(neg_data)
        n = min(len(pos_samples), len(neg_samples))
        random.shuffle(pos_samples)
        random.shuffle(neg_samples)
        split_list.extend(pos_samples[:n])
        split_list.extend(neg_samples[:n])
        random.shuffle(split_list)

    return sft_train, sft_valid, sft_test


print('✅ Tất cả hàm đã định nghĩa xong')
print(f'   prepare_positive_data: có reasoning + confidence filter')
print(f'   shuffle_and_select_data: tỷ lệ 1:{NEG_PER_POS}, cân bằng error type')


✅ Tất cả hàm đã định nghĩa xong


## 5. Kiểm tra nhanh: đọc thử 1 error file

In [ ]:
# Tìm 1 error file bất kỳ để xem cấu trúc
sample_error_file = None
for root, dirs, files in os.walk(ERROR_PATH):
    for f in files:
        if f.endswith('.txt'):
            sample_error_file = os.path.join(root, f)
            break
    if sample_error_file:
        break

if sample_error_file is None:
    print('⚠️  Chưa có error file nào trong short_error_dataset/ — kiểm tra bước 5')
else:
    rel = os.path.relpath(sample_error_file, ERROR_PATH)
    print(f'📄 Sample error file: {rel}\n')

    with open(sample_error_file, 'r', encoding='utf-8') as f:
        sample = eval(f.read())

    print('Keys trong file:')
    for k, v in sample.items():
        print(f'  {k}: {str(v)[:100]}')

    # Thử sinh output
    path_parts = rel.replace('\\', '/').split('/')
    raw_key    = path_parts[-3].lower() if len(path_parts) >= 3 else ''
    matched    = next((v for k, v in ERROR_TYPE_DICT.items() if k in raw_key or raw_key in k), None)
    print(f'\nNhận dạng error type: "{raw_key}" → {matched}')

    if matched:
        sample['error type'] = matched
        sample['method']     = path_parts[-2] if len(path_parts) >= 2 else 'unknown'
        output = generate_negative_output(sample)
        print(f'\n--- Output mẫu ---')
        print(output[:400] + '...' if len(output) > 400 else output)

📄 Sample error file: Suc_khoe_17_người_nghi_ngộ_độc_sau_khi_ăn_bánh_mì_xe_đẩy (1)/structured_extrinsic/extrinsic/extrinsic information/introducing extrinsic information/Introducing extrinsic information.txt

Keys trong file:
  original text in summary: Sáng 24/3, ông Đỗ Ngọc Hòa, Phó giám đốc Sở Y tế Quảng Ngãi, thông báo có người nghi ngộ độc sau khi
  chosen element: Sáng 24/3, ông Đỗ Ngọc Hòa, Phó giám đốc Sở Y tế Quảng Ngãi, thông báo có người nghi ngộ độc sau khi
  modification explanation: Thêm thông tin về việc có nhiều người đã từng bị ngộ độc thực phẩm tại khu vực này trước đó.
  modified element: trong đó có nhiều trường hợp ngộ độc thực phẩm đã xảy ra trước đây.
  modified text: Sáng 24/3, ông Đỗ Ngọc Hòa, Phó giám đốc Sở Y tế Quảng Ngãi, thông báo có người nghi ngộ độc sau khi
  explanation: Thông tin về việc có nhiều trường hợp ngộ độc thực phẩm trước đây không được đề cập trong tài liệu g
  full text of modified summary: ['Sáng 24/3, ông Đỗ Ngọc Hòa, Phó giám 

## 6. Load toàn bộ data & xuất JSONL

> ⏱️ Bước này không gọi API nào, chỉ đọc file từ Drive — thường xong trong vài giây đến vài phút tùy số lượng document.

In [ ]:
# Reset bộ đếm trước khi chạy lại
empty_counter = {k: 0 for k in ERROR_SAMPLE_NUM}
pos_counter = pos_num = 0
neg_counter = neg_num = 0

# ── Load data ─────────────────────────────────────────────────────────
print('=== Bước 1: Load data ===')
full_sft_data = prepare_sft_data()

# ── Chia train/valid/test ─────────────────────────────────────────────
print('\n=== Bước 2: Chia train/valid/test ===')
sft_train, sft_valid, sft_test = shuffle_and_select_data(full_sft_data, round_num=1)

# ── Xuất JSONL ────────────────────────────────────────────────────────
print('\n=== Bước 3: Xuất JSONL ===')

train_data = sft_train[0]
files_to_write = [
    (f'summary_sft_train_pos1neg{NEG_PER_POS}_with_ref.jsonl', train_data),
    ('summary_sft_valid_with_ref.jsonl',          sft_valid),
    ('summary_sft_test_with_ref.jsonl',           sft_test),
]

for fname, data in files_to_write:
    out_path = os.path.join(SFT_PATH, fname)
    with jsonlines.open(out_path, mode='w') as writer:
        for item in data:
            writer.write(item)
    print(f'  ✅ {fname}: {len(data)} samples → {out_path}')

# Xuất thêm 10 mẫu ví dụ để xem thử
example_path = os.path.join(OUTPUT_ROOT, 'example_summary_sft_data_with_ref.jsonl')
with jsonlines.open(example_path, mode='w') as writer:
    for item in train_data[:10]:
        writer.write(item)
print(f'  📝 example (10 mẫu)      → {example_path}')

# ── Thống kê cuối ────────────────────────────────────────────────────
print(f'\n{"="*50}')
print(f'📊 Thống kê:')
print(f'   Train : {len(train_data):>6} samples')
print(f'   Valid : {len(sft_valid):>6} samples')
print(f'   Test  : {len(sft_test):>6} samples')
print(f'   Tổng  : {len(train_data)+len(sft_valid)+len(sft_test):>6} samples')
if pos_num > 0:
    print(f'   Avg positive output length : {pos_counter/pos_num:.1f} từ')
if neg_num > 0:
    print(f'   Avg negative output length : {neg_counter/neg_num:.1f} từ')

=== Bước 1: Load data ===
Đang load 418 supported summary...
  [SKIP] Không tìm thấy document: Cong_nghe_Amazon__chuẩn_bị_quay_lại_thị_trường_smartphone_
  [SKIP] Không tìm thấy document: Cong_nghe_Anthropic,_OpenAI_tuyển_chuyên_gia_ngăn__bom_bẩn__cho_AI
  [SKIP] Không tìm thấy document: Cong_nghe_Apple_sẽ_giới_thiệu_loạt__tiến_bộ_về_AI__ngày_86
  [SKIP] Không tìm thấy document: Cong_nghe_CEO_Cloudflare__Lượng_truy_cập_của_bot_vượt_con_người_năm_2027_
  [SKIP] Không tìm thấy document: Cong_nghe_Chiêu_lừa__hỗ_trợ_lấy_lại_tiền__nở_rộ_sau_vụ_sập_sàn_Onus
  [SKIP] Không tìm thấy document: Cong_nghe_Jensen_Huang__Siêu_trí_tuệ_AGI_đã_xuất_hiện_
  [SKIP] Không tìm thấy document: Cong_nghe_Mark_Zuckerberg_tạo__tác_nhân_CEO__hỗ_trợ_chính_mình
  [SKIP] Không tìm thấy document: Cong_nghe_Tình_trạng_thiếu_hụt_chip_nhớ__kéo_dài_đến_năm_2030_
  [SKIP] Không tìm thấy document: Cong_nghe_iPhone_18_có_thể_hồi

## 7. Xem trước nội dung dataset

In [ ]:
# In 1 positive sample và 1 negative sample để kiểm tra
print('=' * 60)
print('📗 POSITIVE SAMPLE (summary đúng):')
print('=' * 60)
pos_samples = [s for s in train_data if 'Therefore, the answer is NO.' in s['text']]
if pos_samples:
    sample_text = random.choice(pos_samples)['text']
    # Chỉ in phần output (sau end_header)
    parts = sample_text.split('<|end_header_id|>:')
    if len(parts) == 2:
        print('[INPUT] (đã ẩn để gọn)')
        print('[OUTPUT]:')
        print(parts[1][:600])
else:
    print('(Chưa có positive sample trong train set)')

print()
print('=' * 60)
print('📕 NEGATIVE SAMPLE (summary có lỗi):')
print('=' * 60)
neg_samples = [s for s in train_data if 'Therefore, the answer is YES.' in s['text']]
if neg_samples:
    sample_text = random.choice(neg_samples)['text']
    parts = sample_text.split('<|end_header_id|>:')
    if len(parts) == 2:
        print('[INPUT] (đã ẩn để gọn)')
        print('[OUTPUT]:')
        print(parts[1][:600])
else:
    print('(Chưa có negative sample trong train set)')

📗 POSITIVE SAMPLE (summary đúng):
[INPUT] (đã ẩn để gọn)
[OUTPUT]:
Summary sentence 1 is supported by the following sentence:
- Ngày 23/3, Văn phòng Cơ quan Cảnh sát điều tra Công an tỉnh Quảng Ninh cho biết đã hoàn tất kết luận điều tra, đề nghị VKS cùng cấp truy tố ông Hà Đăng Huấn, 61 tuổi, về hai tội Gây rối trật tự công cộng và Lợi dụng các quyền tự do dân chủ xâm phạm lợi ích của Nhà nước, quyền, lợi ích hợp pháp của tổ chức, cá nhân.

Summary sentence 2 is supported by the following sentences:
- Theo điều tra, từ cuối tháng 9/2022 đến tháng 10/2025, ông Huấn thường xuyên trong tình trạng say xỉn, đến trụ sở các cơ quan như Đảng ủy, UBND, công an trên đ

📕 NEGATIVE SAMPLE (summary có lỗi):
[INPUT] (đã ẩn để gọn)
[OUTPUT]:
The following part is not supported by the origin document:
- Location: The error occurs in the following sentence: Công trình đường sách Nguyễn Đổng Chi tại quận 7, TP HCM, được khởi công với mục tiêu tạo không gian cho những người yêu sách, yêu tri thức, kết n

## 8. Phân tích phân phối error type

In [ ]:
from collections import Counter

def count_error_types(data: list) -> Counter:
    c = Counter()
    for sample in data:
        text = sample['text']
        # Chỉ tìm Error Type trong phần OUTPUT (sau <|end_header_id|>:)
        output_part = text.split('<|end_header_id|>:')[-1]
        if 'Therefore, the answer is YES.' in output_part:
            m = re.search(r'- Error Type: (.+)', output_part)
            if m:
                c[m.group(1).strip()] += 1
        elif 'Therefore, the answer is NO.' in output_part:
            c['[Positive — No Error]'] += 1
    return c

for split_name, split_data in [('Train', train_data), ('Valid', sft_valid), ('Test', sft_test)]:
    counts = count_error_types(split_data)
    total  = sum(counts.values())
    print(f'\n📊 {split_name} ({total} samples):')
    for error_type, count in sorted(counts.items()):
        bar = '█' * (count * 30 // max(counts.values()))
        print(f'  {error_type:<30} {count:>4}  {bar}')


📊 Train (264 samples):
  Specify the exact error type based on the categories listed above.  131  █████████████████████████████
  [Positive — No Error]           133  ██████████████████████████████

📊 Valid (194 samples):
  Specify the exact error type based on the categories listed above.   97  ██████████████████████████████
  [Positive — No Error]            97  ██████████████████████████████

📊 Test (200 samples):
  Specify the exact error type based on the categories listed above.  100  ██████████████████████████████
  [Positive — No Error]           100  ██████████████████████████████


## ✅ Hoàn thành Pipeline!

Các file JSONL đã được lưu vào Drive:

```
training_dataset_construct/sft_dataset/jsonl/
  ├── summary_sft_train_pos1neg2_with_ref.jsonl  ← dùng để fine-tune
  ├── summary_sft_valid_with_ref.jsonl           ← dùng để validate
  └── summary_sft_test_with_ref.jsonl            ← dùng để test
```

**Format mỗi sample (positive — có reasoning):**
```
Sentence 1: "<câu tóm tắt>"
→ Supported by: "<câu gốc trong document>"

Sentence 2: "<câu tóm tắt>"
→ Supported by multiple sentences:
  "<câu gốc 1>"
  "<câu gốc 2>"

All sentences in the summary are directly supported by the original document.
Therefore, the answer is NO.
```

**Format mỗi sample (negative):**
```
The following part is not supported by the origin document:
- Location: The error occurs in the following sentence: <câu sai>. Specifically, '<phần sai>'.
- Explanation: <giải thích>
- Correction: <câu đúng>
- Error Type: <loại lỗi>
Therefore, the answer is YES.
```
